In [1]:
pip install pandas beautifulsoup4 lxml konlpy JPype1-py3 fpdf transformers datasets torch sentencepiece



  Using cached sentencepiece-0.2.0-cp310-cp310-win_amd64.whl (991 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
  Using cached frozenlist-1.5.0-cp310-cp310-win_amd64.whl (51 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl (7.6 kB)
  Created wheel for JPype1-py3: filename=JPype1_py3-0.5.5.4-cp310-cp310-win_amd64.whl size=187910 sha256=34fcf37ad1d36eaac5eeb34b7c5cdfbbd967cb48bf937a948aa8584f7fc1492f
  Stored in directory: c:\users\lg\appdata\local\pip\cache\wheels\57\72\ea\b886a286a27c6e3c35ba9e9833b13abc5c5bdc0a9cad91e328
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40730 sha256=d21b6eb87b54dfc4214acfe51ef12d8dfdaf4d5f9213d18fb93c44a22fef32ed
  Stored in directory: c:\users\lg\appdata\local\pip\cache\wheels\f9\95\ba\f418094659025eb9611f17cbcaf2334236bf39a0c3453ea455
Successfully built JPype1-py3 fpdf
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.2.0
    Uninstalling fsspec-2025.2.0:
      Successfully u

You should consider upgrading via the 'c:\Users\LG\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
#데이터 로딩 및 기본 검토
import pandas as pd

df= pd.read_csv('sbs_news_articles.csv', encoding='utf-8')
#컬럼 목록 확인
print(df.columns)
#데이터 샘플 확인
print(df.sample(5))

Index(['제목', '링크', '날짜', '본문'], dtype='object')
                                                   제목  \
48                  한 대행의 '헌법재판관 지명'…모든 권한 행사 가능? 위헌?   
54    [자막뉴스] "65세 넘어도 일 쭉쭉 하세요!"…취업난 속 직원 너무 절실하다는 이곳   
95              '8억대 금품수수' 전준경 전 민주연 부원장, 1심 징역 2년6개월   
43                 미국 관세 폭탄에 맞대응 나선 중국…트럼프-시진핑 '치킨게임'   
85  [김연경 직캠] 중계로는 다 담을 수 없던 '환희의 순간'...마지막까지 완벽했던 ...   

                                                   링크  \
48  https://news.sbs.co.kr/news/endPage.do?news_id...   
54  https://news.sbs.co.kr/news/endPage.do?news_id...   
95  https://news.sbs.co.kr/news/endPage.do?news_id...   
43  https://news.sbs.co.kr/news/endPage.do?news_id...   
85  https://news.sbs.co.kr/news/endPage.do?news_id...   

                                날짜  \
48  Wed, 9 Apr 2025 16:15:00 +0900   
54  Wed, 9 Apr 2025 15:58:00 +0900   
95  Wed, 9 Apr 2025 15:10:00 +0900   
43  Wed, 9 Apr 2025 16:19:00 +0900   
85  Wed, 9 Apr 2025 15:08:00 +0900   

                                     

In [10]:
# HTML 태그 및 불필요한 문자 제거import re
from bs4 import BeautifulSoup
import re

def clean_text(text):
    # 1. HTML 태그 제거
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text()
    # 2. 특수문자 및 불필요한 공백 제거
    text = re.sub(r'\s+', ' ', text)          # 연속된 공백을 하나로
    # 필요에 따라 특정 특수문자나 이모지를 제거
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[▲■<>\[\]]', '', text)  # 알파벳, 숫자, 공백 이외의 문자 제거
    return text.strip()

# 예시: article_body 컬럼에 대해 클리닝 수행 (컬럼 이름이 다르면 수정)
df['본문_문자제거'] = df['본문'].astype(str).apply(clean_text)


#본문 컬럼 확인
print(df[['본문_문자제거','본문']].head(5))

                                             본문_문자제거  \
0  미국 트럼프 대통령의 상호관세 폭탄이 전 세계를 흔들고 있습니다 주가는 곤두박질쳤고...   
1  공사 현장에서 발견된 포탄오늘9일 오후 4시 50분쯤 경북 포항시 북구 학산천 인근...   
2  오늘 놓치지 말아야 할 이슈 퇴근길에 보는 이브닝 브리핑에 있습니다이번 대통령 선거...   
3  어느덧 스위치온 다이어트 2주 차밀가루를 금지당한 루나가 간절히 바라는 것은 다름 ...   
4  미국의 상호관세 발효를 하루 앞두고 트럼프 미국 대통령은 미국이 관세로 매일 20억...   

                                                  본문  
0  미국 트럼프 대통령의 상호관세 폭탄이 전 세계를 흔들고 있습니다. 주가는 곤두박질쳤...  
1  ▲ 공사 현장에서 발견된 포탄오늘(9일) 오후 4시 50분쯤 경북 포항시 북구 학산...  
2  오늘 놓치지 말아야 할 이슈, 퇴근길에 보는 이브닝 브리핑에 있습니다."이번 대통령...  
3  어느덧 스위치온 다이어트 2주 차!밀가루를 금지당한 루나가 간절히 바라는 것은 다름...  
4  미국의 상호관세 발효를 하루 앞두고 트럼프 미국 대통령은 미국이 관세로 매일 20억...  


In [9]:
# 중복된 기사 있는지 찾는 코드
# 중복된 기사 확인 (예: '제목' 컬럼 기준으로 중복 확인)
duplicate_rows = df[df.duplicated(subset=['제목'], keep=False)]

# 중복된 기사 출력
print(f"중복된 기사 수: {len(duplicate_rows)}")
print(duplicate_rows)

중복된 기사 수: 0
Empty DataFrame
Columns: [제목, 링크, 날짜, 본문, 본문_문자제거]
Index: []


In [11]:
#데이터프레임을 JSON으로 변환
import json

import json

# 각 행마다 JSON 객체 생성 (예: 제목, 본문, 날짜, 링크)
train = []
for _, row in df.iterrows():
    sample = {
        "title": row.get("제목", ""),
        "link": row.get("링크", ""),
        "date": row.get("날짜", ""),
        "body": row.get("본문", ""),
        "clean_body": row.get("본문_문자제거", "")
    }
    train.append(sample)

# JSON 파일 저장
with open("sbs_articles.json", "w", encoding="utf-8") as f:
    json.dump(train, f, ensure_ascii=False, indent=2)


In [18]:
#추가 정규화
"""
-띄어쓰기 및 공백 정리: 연속된 공백이나 불필요한 공백 제거
-문장 구분 및 분할: 문장 단위로 분할하여 모델 입력 길이에 맞제 조정 및 나누어 학습 데이터로 사용
-한글 정규화: 한글 자모 분리나, 합치기
"""
import json
json_file_path = 'sbs_articles.json'

with open(json_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

def additional_normalize(text):
    # 예를 들어, 양쪽 공백 제거 및 불필요한 탭/줄바꿈 제거
    text = " ".join(text.split())
    return text

# 각 기사의 본문에 대해 추가 정규화 적용
# clean_body 컬럼에 추가 정규화 적용
df['본문_정규화'] = df['본문_문자제거'].astype(str).apply(additional_normalize)

# 결과 확인
print(df[['본문_정규화', '본문_문자제거']])


                                                본문_정규화  \
0    미국 트럼프 대통령의 상호관세 폭탄이 전 세계를 흔들고 있습니다 주가는 곤두박질쳤고...   
1    공사 현장에서 발견된 포탄오늘9일 오후 4시 50분쯤 경북 포항시 북구 학산천 인근...   
2    오늘 놓치지 말아야 할 이슈 퇴근길에 보는 이브닝 브리핑에 있습니다이번 대통령 선거...   
3    어느덧 스위치온 다이어트 2주 차밀가루를 금지당한 루나가 간절히 바라는 것은 다름 ...   
4    미국의 상호관세 발효를 하루 앞두고 트럼프 미국 대통령은 미국이 관세로 매일 20억...   
..                                                 ...   
96   경영 여건 악화의 영향으로 가맹 브랜드의 수가 조사 시작 이래 처음으로 감소한 것으...   
97   경기 안산상록경찰서는 금은방에서 목걸이를 훔친 혐의로 30대 A 씨를 검거해 조사하...   
98   김신조 목사북한 무장공비로 우리나라에 침투했다가 귀순한 뒤 목회생활을 했던 김신조 ...   
99   배우 김민희가 홍상수 감독의 2세를 출산했다8일 영화계에 따르면 김민희는 최근 아들...   
100  김문수 전 고용노동부 장관이 9일 서울 여의도 국회 소통관에서 제21대 대통령 경선...   

                                               본문_문자제거  
0    미국 트럼프 대통령의 상호관세 폭탄이 전 세계를 흔들고 있습니다 주가는 곤두박질쳤고...  
1    공사 현장에서 발견된 포탄오늘9일 오후 4시 50분쯤 경북 포항시 북구 학산천 인근...  
2    오늘 놓치지 말아야 할 이슈 퇴근길에 보는 이브닝 브리핑에 있습니다이번 대통령 선거...  
3    어느덧 스위치온 다이어트 2주 차밀가루를 금지당한 루나가 간절히 바라는 것은 다름 ...  
4    미국의 상호관세 발효를 

In [21]:
# 원본(clean_body)과 정규화된 데이터(clean_body_normalized) 비교
df['difference'] = df['본문_문자제거'] != df['본문_정규화']

# 차이가 있는 행만 출력
differences = df[df['difference']]
print("정규화로 인해 변경된 데이터:")
print(differences[['본문_문자제거', '본문_정규화']])

정규화로 인해 변경된 데이터:
                                              본문_문자제거  \
0   미국 트럼프 대통령의 상호관세 폭탄이 전 세계를 흔들고 있습니다 주가는 곤두박질쳤고...   
2   오늘 놓치지 말아야 할 이슈 퇴근길에 보는 이브닝 브리핑에 있습니다이번 대통령 선거...   
3   어느덧 스위치온 다이어트 2주 차밀가루를 금지당한 루나가 간절히 바라는 것은 다름 ...   
4   미국의 상호관세 발효를 하루 앞두고 트럼프 미국 대통령은 미국이 관세로 매일 20억...   
5   지난 3일 경북 포항시 저녁 7시쯤 한 여성이 식당 주방으로 들어오더니 집기들을 집...   
8   현지시간 4월 8일 새벽 도미니카공화국의 수도 산토도밍고에 위치한 제트세트 나이트클...   
10  앵커대전의 학교 급식 조리 종사원들의 파업으로 급식의 질이 눈에 띄게 떨어지고 있다...   
12  오늘9일 오전 21대 대선 출마 선언을 위해 국회를 찾은 김문수 전 고용노동부 장관...   
14  앵커제주 신항만 기본 건설 계획이 변경 고시됐습니다 변경 내용에 따르면 제주 신항은...   
15  오 클릭 마지막 검색어는 삽시간에 고속도로 불바다입니다브라질 남부 산타카타리나주의 ...   
16  미국 중부 지역을 강타한 폭풍으로 기록적인 홍수에 이어 토네이도까지 발생해 피해가 ...   
18  오 클릭 두 번째 검색어는 자전거 타다가 넘어지고 화풀이입니다블랙박스 제보 차량이 ...   
20  앵커서울 시민들은 70살은 넘어야 노인이라고 생각하고 정년 연장에 대해서는 10명 ...   
24  앵커여권에서도 출마 선언이 잇따랐습니다 김문수 전 장관이 국민의힘 입당 원서를 내고...   
26  앵커실제로 관세 폭탄이 터지자 오늘9일 우리 금융시장은 또 한번 직격탄을 맞았습니다...   
31  오전 8시 68살 김태호 씨가 도시락 배달에 나섭니다저소득 노인들에게 따뜻한 한 끼...   
41  공연이 열리고 있는

In [23]:
#정규화한 데이터프레임을 JSON으로 다시 변환
import json
# 각 행마다 JSON 객체 생성 (예: 제목, 본문, 날짜, 링크)
train = []
for _, row in df.iterrows():
    sample = {
        "title": row.get("제목", ""),
        "link": row.get("링크", ""),
        "date": row.get("날짜", ""),
        "body": row.get("본문", ""),
        "clean_body": row.get("본문_문자제거", ""),
        "normal_body": row.get("본문_정규화", "")
    }
    train.append(sample)

# JSON 파일 저장
with open("sbs_articles.json", "w", encoding="utf-8") as f:
    json.dump(train, f, ensure_ascii=False, indent=2)


In [2]:
# 필요한 라이브러ㅣ 설치: 깃허브에서 KoBART 설치
# https://github.com/SKT-AI/KoBART
! pip install git+https://github.com/SKT-AI/KoBART#egg=kobart


  Cloning https://github.com/SKT-AI/KoBART to c:\users\lg\appdata\local\temp\pip-install-pzpeuurn\kobart_8f3a98c597e94fbb9dcac8ec7ee2ff06
  Resolved https://github.com/SKT-AI/KoBART to commit eec563bfccf723cae8fd0fff02d5b2b09e847516
  Using cached boto3-1.37.30-py3-none-any.whl (139 kB)
  Using cached pytorch_lightning-1.2.1-py3-none-any.whl (814 kB)


  Running command git clone -q https://github.com/SKT-AI/KoBART 'C:\Users\LG\AppData\Local\Temp\pip-install-pzpeuurn\kobart_8f3a98c597e94fbb9dcac8ec7ee2ff06'
ERROR: Could not find a version that satisfies the requirement torch==1.7.1 (from kobart) (from versions: 1.11.0, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0)
ERROR: No matching distribution found for torch==1.7.1
You should consider upgrading via the 'C:\Users\LG\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
#모델별 토그나이저 처리: 모델-kobart or kot5 사용(지금은 kobart 사용)
from kobart import get_kobart_tokenizer
from trannsformers import BartTokenizer, BartForConditionalGeneration
import torch

#json 데이터 파일 경로 지정
json_file_path = 'dataset/sbs_articles.json'
#json 파일 로드
with open(json_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
tokenizer = get_kobart_tokenizer()

def preprocess_for_kobart(input_text,max_length=512):
    # KoBART 토크나이저로 인코딩
    encoding=tokenizer(
        input_text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encoding

#데이터 인코딩
train=[]
for sample in data:
    input_text = sample.get("normal_body", "")
    encoding = preprocess_for_kobart(input_text)
    train.append(encoding)

#결과 확인
print(f"인코딩된 데이터 샘플 수: {len(train)}")

ModuleNotFoundError: No module named 'kobart'